In [ ]:
!pip -q install pyarrow

from google.colab import drive
from pathlib import Path
from IPython.display import display

import json
import unicodedata
import numpy as np
import pandas as pd

drive.mount("/content/drive")

INPUT_PATH = (
    "/content/drive/MyDrive/KB_FinalProject_Team6/"
    "완전쓰는거_ver2/인허가_master.csv"
)

START_MONTH = "2023-01-01"
END_MONTH = "2026-04-01"

# 실제 데이터 관측 완료일을 확인한 경우에만 입력하세요.
# 예: OBSERVED_THROUGH = "2026-10-15"
# None이면 전체 피처는 생성하고 폐업 라벨은 결측으로 저장합니다.
OBSERVED_THROUGH = None

# 명세서에 표본 부족 임계값이 없어 기본 5개로 설정
SMALL_N = 5
PRIOR_WEIGHT = 5

# 명세서에 기재된 폐업 기록 시작일
CLOSE_HISTORY_START = pd.Timestamp("2022-01-01")

# 월수 환산 기준: 일수 / 평균 월 길이
DAYS_PER_MONTH = 365.25 / 12

months = pd.date_range(
    start=START_MONTH,
    end=END_MONTH,
    freq="MS",
)

print(
    f"계산 기간: {months[0]:%Y-%m} ~ "
    f"{months[-1]:%Y-%m}, 총 {len(months)}개월"
)

In [ ]:
def resolve_korean_path(path):
    target = Path(path)

    if target.exists():
        return target

    current = Path(target.anchor)

    for part in target.parts[1:]:
        direct = current / part

        if direct.exists():
            current = direct
            continue

        normalized = unicodedata.normalize("NFC", part)

        matches = [
            item
            for item in current.iterdir()
            if unicodedata.normalize("NFC", item.name) == normalized
        ]

        if len(matches) != 1:
            raise FileNotFoundError(
                f"경로를 확인하세요: {current / part}"
            )

        current = matches[0]

    return current


csv_path = resolve_korean_path(INPUT_PATH)

for encoding in ["utf-8-sig", "cp949"]:
    try:
        raw = pd.read_csv(
            csv_path,
            encoding=encoding,
            dtype="string",
        )
        break
    except UnicodeDecodeError:
        continue
else:
    raise ValueError("CSV 인코딩을 확인하세요.")

OUT = csv_path.parent / "인허가_파생지표_202301_202604"
SNAPSHOT_DIR = OUT / "snapshots"

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

print("입력 파일:", csv_path)
print("원본 크기:", raw.shape)
print("결과 저장:", OUT)

display(raw.head(3))

In [ ]:
required = [
    "점포이력ID",
    "행정동코드",
    "구군",
    "업종명",
    "카드업종",
    "프랜차이즈유무",
    "인허가일자_dt",
    "폐업일자_dt",
]

missing = set(required) - set(raw.columns)

if missing:
    raise ValueError(f"필수 열 누락: {sorted(missing)}")

if raw["점포이력ID"].isna().any():
    raise ValueError("점포이력ID 결측이 있습니다.")

if raw["점포이력ID"].duplicated().any():
    raise ValueError(
        "점포이력ID가 중복됩니다. "
        "이 코드는 점포이력ID당 한 행인 통합본을 대상으로 합니다."
    )

d = raw[required].copy()

for source, target in [
    ("인허가일자_dt", "open_s"),
    ("폐업일자_dt", "close_s"),
]:
    text = d[source].str.strip()

    d[target] = pd.to_datetime(
        text,
        format="%Y-%m-%d",
        errors="coerce",
    )

    invalid = (
        text.notna()
        & text.ne("")
        & d[target].isna()
    )

    if invalid.any():
        display(raw.loc[invalid, [source]].head(10))
        raise ValueError(
            f"{source} 날짜 변환 실패: {invalid.sum():,}건"
        )

# 신호동 → 녹산동
d["행정동코드"] = (
    d["행정동코드"]
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .replace({"2644059000": "2644056000"})
)

d["구군"] = d["구군"].str.strip()
d["업종명"] = d["업종명"].str.strip()

d["카드업종"] = (
    d["카드업종"]
    .str.strip()
    .replace({"음식료픔": "음식료품"})
)

franchise_text = d["프랜차이즈유무"].str.strip()

d["is_franchise"] = pd.to_numeric(
    franchise_text,
    errors="coerce",
).astype(float)

invalid_franchise = (
    franchise_text.fillna("").ne("")
    & (
        d["is_franchise"].isna()
        | ~d["is_franchise"].isin([0, 1])
    )
)

if invalid_franchise.any():
    display(
        raw.loc[
            invalid_franchise,
            ["점포이력ID", "프랜차이즈유무"]
        ].head(10)
    )
    raise ValueError("프랜차이즈유무에 0·1 이외 값이 있습니다.")

reasons = pd.Series("", index=d.index, dtype="string")


def add_reason(mask, reason):
    mask = mask.fillna(False)
    reasons.loc[mask] = reasons.loc[mask] + reason + ";"


add_reason(d["open_s"].isna(), "개업일 없음")
add_reason(
    d["close_s"].lt(d["open_s"]),
    "폐업일이 개업일보다 빠름",
)

for col in ["행정동코드", "구군", "업종명"]:
    add_reason(
        d[col].fillna("").str.strip().eq(""),
        f"{col} 없음",
    )

# 현재 상태는 품질 점검에만 사용
if "폐업유무_기록기준" in raw.columns:
    closed_flag = pd.to_numeric(
        raw["폐업유무_기록기준"],
        errors="coerce",
    ).eq(1)

    add_reason(
        closed_flag & d["close_s"].isna(),
        "폐업 기록은 있으나 폐업일 없음",
    )

if "영업여부" in raw.columns:
    closed_status = (
        raw["영업여부"]
        .str.strip()
        .str.upper()
        .eq("F")
        .fillna(False)
    )

    add_reason(
        closed_status & d["close_s"].isna(),
        "폐업 상태이나 폐업일 없음",
    )

if "보류사유" in raw.columns:
    add_reason(
        raw["보류사유"].fillna("").str.strip().ne(""),
        "통합 판단 보류",
    )

excluded_mask = reasons.ne("")

excluded = raw.loc[excluded_mask].copy()
excluded["파생지표_제외사유"] = reasons.loc[excluded_mask]

excluded.to_csv(
    OUT / "excluded_rows.csv",
    index=False,
    encoding="utf-8-sig",
)

d = (
    d.loc[~excluded_mask]
    .drop(
        columns=[
            "인허가일자_dt",
            "폐업일자_dt",
            "프랜차이즈유무",
        ]
    )
    .reset_index(drop=True)
)

d.to_parquet(
    OUT / "store_master.parquet",
    index=False,
)

print(f"원본 점포: {len(raw):,}개")
print(f"계산 대상: {len(d):,}개")
print(f"제외 점포: {len(excluded):,}개")
print(f"행정동: {d['행정동코드'].nunique():,}개")

display(d.head())

In [ ]:
def age_months(end, start):
    return (
        (end - start).dt.total_seconds()
        / (86400 * DAYS_PER_MONTH)
    )


def active(data, t):
    """명세서의 기준월 영업 점포 정의."""
    return (
        data["open_s"].lt(t)
        & (
            data["close_s"].isna()
            | data["close_s"].ge(t)
        )
    )


def lookup(rows, source, keys, value=None, how="size"):
    """집계 결과를 rows의 행 순서에 맞게 반환."""
    grouped = source.groupby(
        keys,
        observed=True,
        dropna=False,
    )

    result = (
        grouped.size()
        if value is None
        else grouped[value].agg(how)
    )

    index = pd.MultiIndex.from_frame(rows[keys])

    return pd.Series(
        result.reindex(index).to_numpy(),
        index=rows.index,
        dtype=float,
    )


def loo_median(rows, keys, value):
    """각 점포 자신을 제외한 그룹 중앙값."""
    result = pd.Series(np.nan, index=rows.index)

    for _, group in rows.groupby(
        keys,
        observed=True,
        sort=False,
    ):
        group = group.sort_values(value, kind="stable")
        values = group[value].to_numpy()
        n = len(values)

        if n < 2:
            continue

        ranks = np.arange(n)
        left = (n - 2) // 2
        right = (n - 1) // 2

        medians = (
            values[left + (ranks <= left)]
            + values[right + (ranks <= right)]
        ) / 2

        result.loc[group.index] = medians

    return result

In [ ]:
def build_month(d, t, observed_through=None):
    t6 = t - pd.DateOffset(months=6)
    t12 = t - pd.DateOffset(months=12)

    # 기준월에 영업 중인 점포만 포함
    x = d.loc[active(d, t)].copy()

    x["base_ym"] = t.strftime("%Y-%m")
    x["age_m"] = age_months(t, x["open_s"])

    x["age_band"] = pd.cut(
        x["age_m"],
        bins=[0, 12, 18, 24, 36, np.inf],
        labels=["0–12", "12–18", "18–24", "24–36", "36+"],
        right=False,
    )

    x["is_new"] = x["age_m"].lt(12).astype("int8")
    x["biz_type"] = x["업종명"]

    # 동네 업력 중앙값: 자신 제외
    x["dong_age_med"] = loo_median(
        x, ["행정동코드"], "age_m"
    )

    x["n_dong_age"] = (
        x.groupby("행정동코드")["open_s"]
        .transform("size")
        - 1
    )

    opened = d.loc[
        d["open_s"].ge(t6)
        & d["open_s"].lt(t)
    ]

    closed = d.loc[
        d["close_s"].ge(t6)
        & d["close_s"].lt(t)
    ]

    previous_active = d.loc[active(d, t6)]

    # 최근 12개월 폐업 점포
    life = d.loc[
        d["close_s"].ge(t12)
        & d["close_s"].lt(t)
    ].copy()

    life["life_m"] = age_months(
        life["close_s"], life["open_s"]
    )

    # t-12개월 이상, t-6개월 미만에 개업한 코호트
    cohort = d.loc[
        d["open_s"].ge(t12)
        & d["open_s"].lt(t6)
    ].copy()

    anniversary = (
        cohort["open_s"] + pd.DateOffset(months=6)
    )

    cohort["survived"] = (
        cohort["close_s"].isna()
        | cohort["close_s"].gt(anniversary)
    ).astype(float)

    # h: 행정동×업종, g: 구군×업종
    for level, keys in [
        ("h", ["행정동코드", "업종명"]),
        ("g", ["구군", "업종명"]),
    ]:
        # 현재 동종 점포 수: 자신 제외
        n = lookup(x, x, keys).fillna(0) - 1
        x[f"n_{level}k"] = n

        # 프랜차이즈 비중
        franchise_sum = (
            lookup(
                x, x, keys,
                "is_franchise", "sum"
            )
            - x["is_franchise"]
        )

        franchise_n = (
            lookup(
                x, x, keys,
                "is_franchise", "count"
            )
            - x["is_franchise"].notna().astype(int)
        )

        x[f"n_fr_{level}"] = franchise_n

        x[f"fr_share_{level}_raw"] = (
            franchise_sum
            / franchise_n.replace(0, np.nan)
        )

        # 6개월 전 영업점 수: 자신이 있었으면 제외
        denominator = (
            lookup(
                x, previous_active, keys
            ).fillna(0)
            - active(x, t6).astype(int)
        )

        # 최근 개업점 수: 자신이 포함되면 제외
        n_open = (
            lookup(x, opened, keys).fillna(0)
            - x["open_s"].ge(t6).astype(int)
        )

        # 현재 영업점 자신은 과거 폐업 집합에 없음
        n_close = lookup(
            x, closed, keys
        ).fillna(0)

        x[f"n_start_{level}k"] = denominator
        x[f"n_open_{level}k"] = n_open
        x[f"n_close_{level}k"] = n_close

        x[f"close_rate_6m_{level}_raw"] = (
            n_close
            / denominator.replace(0, np.nan)
        )

        x[f"open_rate_6m_{level}_raw"] = (
            n_open
            / denominator.replace(0, np.nan)
        )

        x[f"net_6m_{level}_raw"] = (
            n_open - n_close
        )

        # 폐업점포 평균 영업기간
        x[f"n_closed_life_{level}"] = (
            lookup(x, life, keys).fillna(0)
        )

        x[f"closed_life_avg_{level}_raw"] = lookup(
            x, life, keys, "life_m", "mean"
        )

        # 생존율 코호트: 자신 제외
        own_cohort = (
            x["open_s"].ge(t12)
            & x["open_s"].lt(t6)
        ).astype(int)

        cohort_n = (
            lookup(x, cohort, keys).fillna(0)
            - own_cohort
        )

        # 현재 영업 중인 자신은 코호트에 속하면 생존자
        survivor_n = (
            lookup(
                x, cohort, keys,
                "survived", "sum"
            ).fillna(0)
            - own_cohort
        )

        x[f"n_cohort_{level}"] = cohort_n

        x[f"surv_6m_{level}_raw"] = (
            survivor_n
            / cohort_n.replace(0, np.nan)
        )

    # 표본 부족 보정
    for feature in [
        "fr_share",
        "close_rate_6m",
        "open_rate_6m",
        "net_6m",
    ]:
        dong_value = x[f"{feature}_h_raw"]
        gu_value = x[f"{feature}_g_raw"]

        corrected = (
            dong_value * x["n_hk"]
            + gu_value * PRIOR_WEIGHT
        ) / (x["n_hk"] + PRIOR_WEIGHT)

        x[feature] = dong_value.where(
            x["n_hk"].ge(SMALL_N),
            corrected,
        )

    # 동종 폐업 5건 미만이면 구군 평균 사용
    x["closed_life_avg"] = (
        x["closed_life_avg_h_raw"].where(
            x["n_closed_life_h"].ge(5),
            x["closed_life_avg_g_raw"],
        )
    )

    x["surv_6m"] = x["surv_6m_h_raw"]

    # 업종 다양성: 자신 제외
    counts = (
        x.groupby(
            ["행정동코드", "업종명"]
        )["open_s"]
        .transform("size")
        .astype(float)
    )

    total = (
        x.groupby("행정동코드")["open_s"]
        .transform("size")
        .astype(float)
    )

    terms = counts * np.log(counts)

    sum_terms = (
        (terms / counts)
        .groupby(x["행정동코드"])
        .transform("sum")
    )

    remaining = counts - 1

    remaining_term = (
        remaining
        * np.log(
            remaining.where(remaining.gt(0), 1)
        )
    )

    x["biz_entropy"] = (
        np.log(
            (total - 1).where(total.gt(1))
        )
        - (
            sum_terms - terms + remaining_term
        ) / (total - 1).replace(0, np.nan)
    )

    x["n_entropy"] = total - 1

    # 명세서의 지표 시작월 적용
    if t < pd.Timestamp("2023-07-01"):
        x[
            [
                "close_rate_6m",
                "open_rate_6m",
                "net_6m",
            ]
        ] = np.nan

    if t < pd.Timestamp("2024-01-01"):
        x[
            [
                "closed_life_avg",
                "surv_6m",
            ]
        ] = np.nan

    # 폐업 이력의 관측 범위 적용
    if t12 < CLOSE_HISTORY_START:
        x["closed_life_avg"] = np.nan

    if t6 < CLOSE_HISTORY_START:
        x[
            ["close_rate_6m", "net_6m"]
        ] = np.nan

    # 6개월 내 폐업 라벨
    label_end = t + pd.DateOffset(months=6)

    x["label_close_6m"] = pd.Series(
        pd.NA,
        index=x.index,
        dtype="Int8",
    )

    if (
        observed_through is not None
        and (
            label_end - pd.Timedelta(days=1)
            <= observed_through
        )
    ):
        x["label_close_6m"] = (
            x["close_s"].ge(t)
            & x["close_s"].lt(label_end)
        ).astype("Int8")

    return x

In [ ]:
observed = (
    pd.Timestamp(OBSERVED_THROUGH)
    if OBSERVED_THROUGH is not None
    else None
)

model_features = [
    "age_m",
    "age_band",
    "is_new",
    "biz_type",
    "is_franchise",
    "dong_age_med",
    "closed_life_avg",
    "surv_6m",
    "fr_share",
    "close_rate_6m",
    "open_rate_6m",
    "net_6m",
    "biz_entropy",
]

summaries = []
missingness = []

# 계산 조건도 함께 저장
settings = {
    "input_path": str(csv_path),
    "start_month": START_MONTH,
    "end_month": END_MONTH,
    "observed_through": OBSERVED_THROUGH,
    "small_n": SMALL_N,
    "prior_weight": PRIOR_WEIGHT,
    "close_history_start": str(CLOSE_HISTORY_START.date()),
    "days_per_month": DAYS_PER_MONTH,
    "model_features": model_features,
    "limitations": [
        "카드·생활인구·면적 등 외부 자료 필요 지표는 미포함",
        "통합본의 대표 업종·프랜차이즈를 과거에도 고정 적용",
        "관측 종료일 미설정 시 라벨은 결측",
        "제외 점포는 집계 모집단에서도 제외",
    ],
}

with open(
    OUT / "calculation_settings.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        settings,
        f,
        ensure_ascii=False,
        indent=2,
    )

for i, t in enumerate(months, start=1):
    x = build_month(d, t, observed)

    # 기본 검증
    assert not x["점포이력ID"].duplicated().any()
    assert x["age_m"].ge(0).all()
    assert x["n_hk"].ge(0).all()

    for col in ["fr_share", "surv_6m"]:
        assert (
            x[col].dropna().between(0, 1).all()
        ), f"{t:%Y-%m}: {col} 범위 오류"

    assert (
        x["biz_entropy"]
        .dropna()
        .ge(-1e-10)
        .all()
    )

    # 폐업률은 명세서 분모 정의상 1을 넘을 수 있으므로 자르지 않음
    path = SNAPSHOT_DIR / f"snapshot_{t:%Y%m}.parquet"

    x.to_parquet(
        path,
        index=False,
        compression="snappy",
    )

    label_rate = x["label_close_6m"].mean()
    n_labels = int(x["label_close_6m"].count())

    summaries.append({
        "base_ym": t.strftime("%Y-%m"),
        "n_stores": len(x),
        "n_labels": n_labels,
        "n_positive": (
            int(x["label_close_6m"].sum())
            if n_labels > 0
            else np.nan
        ),
        "positive_rate": (
            float(label_rate)
            if pd.notna(label_rate)
            else np.nan
        ),
        "close_rate_over_1": int(
            x["close_rate_6m"].gt(1).sum()
        ),
    })

    month_missing = (
        x[model_features]
        .isna()
        .mean()
        .rename("missing_rate")
        .rename_axis("feature")
        .reset_index()
    )

    month_missing.insert(
        0,
        "base_ym",
        t.strftime("%Y-%m"),
    )

    missingness.append(month_missing)

    # 중간 요약도 매달 저장
    pd.DataFrame(summaries).to_csv(
        OUT / "monthly_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )

    pd.concat(
        missingness,
        ignore_index=True,
    ).to_csv(
        OUT / "feature_missingness.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"[{i:02d}/{len(months)}] "
        f"{t:%Y-%m} | "
        f"점포 {len(x):,}개 | "
        f"라벨 {n_labels:,}개 | 저장 완료",
        flush=True,
    )

    del x

summary = pd.DataFrame(summaries)
missing_summary = pd.concat(
    missingness,
    ignore_index=True,
)

print("\n전체 기간 계산 완료")
print("총 스냅샷 행 수:", f"{summary['n_stores'].sum():,}")
print("저장 위치:", OUT)

display(summary)

In [ ]:
audit_rows = []

for t in months:
    path = SNAPSHOT_DIR / f"snapshot_{t:%Y%m}.parquet"

    if not path.exists():
        audit_rows.append({
            "base_ym": t.strftime("%Y-%m"),
            "file_exists": False,
        })
        continue

    temp = pd.read_parquet(path)

    row = {
        "base_ym": t.strftime("%Y-%m"),
        "file_exists": True,
        "n_stores": len(temp),
        "duplicate_store_ids": int(
            temp["점포이력ID"].duplicated().sum()
        ),
        "invalid_open_date": int(
            temp["open_s"].ge(t).sum()
        ),
        "already_closed": int(
            temp["close_s"].lt(t).sum()
        ),
        "n_labels": int(
            temp["label_close_6m"].count()
        ),
        "label_positive_rate": (
            float(temp["label_close_6m"].mean())
            if temp["label_close_6m"].notna().any()
            else np.nan
        ),
        "close_rate_over_1_share": float(
            temp["close_rate_6m"].gt(1).mean()
        ),
        "small_group_share": float(
            temp["n_hk"].lt(SMALL_N).mean()
        ),
        "zero_start_group_share": float(
            temp["n_start_hk"].eq(0).mean()
        ),
    }

    for col in model_features:
        row[f"missing__{col}"] = float(
            temp[col].isna().mean()
        )

    audit_rows.append(row)
    del temp

audit = pd.DataFrame(audit_rows)

audit.to_csv(
    OUT / "validation_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

display(audit[[
    "base_ym",
    "file_exists",
    "n_stores",
    "duplicate_store_ids",
    "invalid_open_date",
    "already_closed",
    "n_labels",
    "label_positive_rate",
    "close_rate_over_1_share",
    "small_group_share",
]])

print("검증 결과 저장:", OUT / "validation_summary.csv")

In [ ]:
import matplotlib.pyplot as plt

plot_data = audit.loc[audit["file_exists"].eq(True)].copy()
plot_data["month"] = pd.to_datetime(plot_data["base_ym"])

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(
    plot_data["month"],
    plot_data["n_stores"],
    marker="o",
    markersize=3,
)
axes[0].set_title("Active stores by base month")
axes[0].set_ylabel("Stores")
axes[0].grid(alpha=0.3)

axes[1].plot(
    plot_data["month"],
    plot_data["close_rate_over_1_share"],
    label="Close rate > 1",
)
axes[1].plot(
    plot_data["month"],
    plot_data["small_group_share"],
    label=f"Group size < {SMALL_N}",
)
axes[1].set_ylabel("Share")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

missing_cols = [
    f"missing__{col}" for col in model_features
]

display(
    plot_data.set_index("base_ym")[missing_cols]
    .rename(columns=lambda name: name.replace("missing__", ""))
    .round(3)
)

In [ ]:
from sklearn.neighbors import BallTree

# 기존 셀에서 만든 점포 기준표 사용
stores = pd.read_parquet(OUT / "store_master.parquet")

required_coords = ["점포이력ID", "lon", "lat"]

missing = set(required_coords) - set(raw.columns)
if missing:
    raise ValueError(f"좌표 계산 필수 열 누락: {sorted(missing)}")

coords = raw[required_coords].copy()

if coords["점포이력ID"].duplicated().any():
    raise ValueError("좌표 원본의 점포이력ID가 중복됩니다.")

for col in ["lon", "lat"]:
    coords[col] = pd.to_numeric(
        coords[col],
        errors="coerce",
    ).astype(float)

# 결측·무한대·경위도 범위 오류·(0, 0) 좌표 제외
valid_coord = (
    np.isfinite(coords["lon"])
    & np.isfinite(coords["lat"])
    & coords["lon"].between(-180, 180)
    & coords["lat"].between(-90, 90)
    & ~(
        coords["lon"].eq(0)
        & coords["lat"].eq(0)
    )
)

coords["coord_valid"] = valid_coord

coords.loc[
    ~coords["coord_valid"],
    ["lon", "lat"],
] = np.nan

# 기존 계산에서 제외한 점포는 공간 집계에서도 제외
stores = stores.merge(
    coords,
    on="점포이력ID",
    how="left",
    validate="one_to_one",
)

stores["coord_valid"] = (
    stores["coord_valid"].fillna(False).astype(bool)
)

coord_report = pd.DataFrame({
    "item": [
        "계산 대상 점포",
        "유효 좌표 점포",
        "좌표 결측 또는 오류 점포",
    ],
    "count": [
        len(stores),
        int(stores["coord_valid"].sum()),
        int((~stores["coord_valid"]).sum()),
    ],
})

coord_report.to_csv(
    OUT / "coordinate_quality_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

stores.loc[
    ~stores["coord_valid"],
    ["점포이력ID", "lon", "lat"],
].to_csv(
    OUT / "invalid_coordinate_stores.csv",
    index=False,
    encoding="utf-8-sig",
)

display(coord_report)

print("유효 좌표 범위")
display(
    stores.loc[
        stores["coord_valid"],
        ["lon", "lat"],
    ].agg(["min", "max"])
)

In [ ]:
EARTH_RADIUS_M = 6_371_008.8


def radians_coords(frame):
    # BallTree의 haversine 입력 순서: 위도, 경도
    return np.radians(
        frame[["lat", "lon"]].to_numpy(dtype=float)
    )


def calculate_spatial_features(snapshot, stores, t):
    # 재실행 시 기존 공간 변수와 좌표를 제거하고 다시 계산
    x = snapshot.drop(
        columns=[
            "lon",
            "lat",
            "coord_valid",
            "same_300m",
            "close_100m_6m",
        ],
        errors="ignore",
    ).copy()

    x = x.merge(
        stores[
            ["점포이력ID", "lon", "lat", "coord_valid"]
        ],
        on="점포이력ID",
        how="left",
        validate="one_to_one",
    )

    x["coord_valid"] = (
        x["coord_valid"].fillna(False).astype(bool)
    )

    same_counts = pd.Series(
        np.nan,
        index=x.index,
        dtype=float,
    )

    closed_counts = pd.Series(
        np.nan,
        index=x.index,
        dtype=float,
    )

    valid_active = x.loc[x["coord_valid"]]

    # 1. 반경 300m 동종 영업점 수
    # 행정동 경계에 관계없이 주변 점포 검색
    for _, group in valid_active.groupby(
        "업종명",
        sort=False,
        observed=True,
    ):
        points = radians_coords(group)

        tree = BallTree(
            points,
            metric="haversine",
        )

        counts = tree.query_radius(
            points,
            r=300 / EARTH_RADIUS_M,
            count_only=True,
        )

        # 검색 결과에 포함된 자기 자신 1개 제외
        same_counts.loc[group.index] = counts - 1

    # 2. 반경 100m 최근 6개월 폐업 수: 모든 업종
    start = t - pd.DateOffset(months=6)

    if start >= CLOSE_HISTORY_START:
        recent_closed = stores.loc[
            stores["coord_valid"]
            & stores["close_s"].ge(start)
            & stores["close_s"].lt(t)
        ]

        if len(valid_active) > 0:
            if recent_closed.empty:
                closed_counts.loc[valid_active.index] = 0
            else:
                closed_tree = BallTree(
                    radians_coords(recent_closed),
                    metric="haversine",
                )

                counts = closed_tree.query_radius(
                    radians_coords(valid_active),
                    r=100 / EARTH_RADIUS_M,
                    count_only=True,
                )

                closed_counts.loc[valid_active.index] = counts

        # 기준월 영업점 자신은 과거 폐업 집합에 없으므로
        # 이 변수에서는 추가로 1을 빼지 않음

    # 결측을 유지할 수 있는 정수형
    x["same_300m"] = same_counts.astype("Int32")
    x["close_100m_6m"] = closed_counts.astype("Int32")

    return x

In [ ]:
spatial_summaries = []

for i, t in enumerate(months, start=1):
    path = SNAPSHOT_DIR / f"snapshot_{t:%Y%m}.parquet"

    if not path.exists():
        raise FileNotFoundError(
            f"기존 월별 결과가 없습니다: {path}"
        )

    snapshot = pd.read_parquet(path)
    original_count = len(snapshot)

    updated = calculate_spatial_features(
        snapshot,
        stores,
        t,
    )

    # 검증
    assert len(updated) == original_count
    assert not updated["점포이력ID"].duplicated().any()

    for col in ["same_300m", "close_100m_6m"]:
        assert updated[col].dropna().ge(0).all(), col

    assert updated.loc[
        ~updated["coord_valid"],
        ["same_300m", "close_100m_6m"],
    ].isna().all().all()

    # 공간 변수를 추가한 월별 결과 저장
    updated.to_parquet(
        path,
        index=False,
        compression="snappy",
    )

    spatial_summaries.append({
        "base_ym": t.strftime("%Y-%m"),
        "n_stores": len(updated),
        "n_valid_coords": int(
            updated["coord_valid"].sum()
        ),
        "same_300m_missing_rate": float(
            updated["same_300m"].isna().mean()
        ),
        "same_300m_mean": (
            float(updated["same_300m"].mean())
            if updated["same_300m"].notna().any()
            else np.nan
        ),
        "same_300m_max": (
            int(updated["same_300m"].max())
            if updated["same_300m"].notna().any()
            else np.nan
        ),
        "close_100m_6m_missing_rate": float(
            updated["close_100m_6m"].isna().mean()
        ),
        "close_100m_6m_mean": (
            float(updated["close_100m_6m"].mean())
            if updated["close_100m_6m"].notna().any()
            else np.nan
        ),
    })

    # 매달 중간 요약 저장
    pd.DataFrame(spatial_summaries).to_csv(
        OUT / "spatial_feature_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"[{i:02d}/{len(months)}] {t:%Y-%m} | "
        f"{len(updated):,}개 점포 | 공간 변수 저장 완료",
        flush=True,
    )

    del snapshot, updated

# 모델 피처 목록 갱신
model_features = list(dict.fromkeys(
    model_features + [
        "same_300m",
        "close_100m_6m",
    ]
))

# 계산 설정 갱신
settings_path = OUT / "calculation_settings.json"

if settings_path.exists():
    with open(settings_path, encoding="utf-8") as f:
        settings = json.load(f)
else:
    settings = {}

settings["model_features"] = model_features

settings["spatial_features"] = {
    "distance": "haversine",
    "earth_radius_m": EARTH_RADIUS_M,
    "same_300m": "同업종 영업점, 300m 이내, 자신 제외",
    "close_100m_6m": "모든 업종, 100m 이내, 최근 6개월 폐업",
    "invalid_coordinates": "NaN",
    "coordinate_time_assumption": "통합본의 좌표를 과거 기준월에도 적용",
}

with open(settings_path, "w", encoding="utf-8") as f:
    json.dump(
        settings,
        f,
        ensure_ascii=False,
        indent=2,
    )

spatial_summary = pd.DataFrame(spatial_summaries)

print("\n40개월 공간 변수 추가 완료")
print("저장 폴더:", SNAPSHOT_DIR)

display(spatial_summary)

sample = pd.read_parquet(
    SNAPSHOT_DIR / "snapshot_202603.parquet"
)

display(
    sample[
        [
            "base_ym",
            "점포이력ID",
            "업종명",
            "coord_valid",
            "same_300m",
            "close_100m_6m",
        ]
    ].head(10)
)

In [ ]:
from google.colab import drive
from pathlib import Path
import unicodedata
import pandas as pd

drive.mount("/content/drive")

def resolve_korean_path(path):
    target = Path(path)
    current = Path(target.anchor)

    for part in target.parts[1:]:
        direct = current / part

        if direct.exists():
            current = direct
            continue

        matches = [
            item for item in current.iterdir()
            if unicodedata.normalize("NFC", item.name)
            == unicodedata.normalize("NFC", part)
        ]

        if len(matches) != 1:
            raise FileNotFoundError(f"경로 확인 필요: {current / part}")

        current = matches[0]

    return current

OUT = resolve_korean_path(
    "/content/drive/MyDrive/KB_FinalProject_Team6/"
    "완전쓰는거_ver2/인허가_파생지표_202301_202604"
)

SNAPSHOT_DIR = OUT / "snapshots"

expected = [
    SNAPSHOT_DIR / f"snapshot_{month:%Y%m}.parquet"
    for month in pd.date_range(
        "2023-01-01", "2026-04-01", freq="MS"
    )
]

missing = [path.name for path in expected if not path.exists()]

print("결과 폴더:", OUT)
print(f"월별 파일: {40 - len(missing)}/40개")

if missing:
    print("누락된 파일:", missing)
else:
    print("40개월 파일이 모두 있습니다. CSV 합치기 코드를 실행하세요.")

merged_csv = OUT / "인허가_파생지표_전체_202301_202604.csv"

if merged_csv.exists():
    print("이미 합친 CSV도 있습니다:", merged_csv)
    print(f"크기: {merged_csv.stat().st_size / 1024**2:,.1f} MB")

In [ ]:
from pathlib import Path
import pandas as pd

# 기존 코드의 OUT 폴더 사용
output_csv = OUT / "인허가_파생지표_전체_202301_202604.csv"
temp_csv = output_csv.with_suffix(".csv.part")

months_to_merge = pd.date_range(
    "2023-01-01", "2026-04-01", freq="MS"
)

paths = [
    OUT / "snapshots" / f"snapshot_{month:%Y%m}.parquet"
    for month in months_to_merge
]

# 40개 파일이 모두 있는지 먼저 확인
missing = [str(path) for path in paths if not path.exists()]

if missing:
    raise FileNotFoundError(
        "누락된 파일이 있습니다:\n" + "\n".join(missing)
    )

total_rows = 0
expected_columns = None

# 임시 파일에 저장하고, 모두 성공하면 최종 파일로 변경
with open(temp_csv, "w", encoding="utf-8-sig", newline="") as f:
    for i, path in enumerate(paths, start=1):
        df = pd.read_parquet(path)

        if expected_columns is None:
            expected_columns = df.columns.tolist()
        else:
            if set(df.columns) != set(expected_columns):
                raise ValueError(
                    f"열 구성이 다릅니다: {path.name}\n"
                    f"누락: {set(expected_columns) - set(df.columns)}\n"
                    f"추가: {set(df.columns) - set(expected_columns)}"
                )

            df = df[expected_columns]

        df.to_csv(
            f,
            index=False,
            header=(i == 1),
        )

        total_rows += len(df)

        print(
            f"[{i:02d}/40] {path.name} | "
            f"{len(df):,}행 | 누적 {total_rows:,}행",
            flush=True,
        )

        del df

temp_csv.replace(output_csv)

print("\n저장 완료")
print("파일:", output_csv)
print(f"전체 행 수: {total_rows:,}")
print(f"파일 크기: {output_csv.stat().st_size / 1024**2:,.1f} MB")

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

paths = [
    OUT / "snapshots" / f"snapshot_{month:%Y%m}.parquet"
    for month in pd.date_range(
        "2023-01-01", "2026-04-01", freq="MS"
    )
]

missing = [str(path) for path in paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "누락된 파일:\n" + "\n".join(missing)
    )

output_path = OUT / "인허가_파생지표_전체_202301_202604.parquet"
temp_path = output_path.with_suffix(".parquet.part")

# 월별 열 구성 확인 및 범주형을 문자열로 통일
def normalize_table(path):
    table = pq.read_table(path)
    fields = []

    for field in table.schema:
        dtype = field.type

        if pa.types.is_dictionary(dtype):
            dtype = dtype.value_type

        fields.append(
            pa.field(field.name, dtype, nullable=True)
        )

    return table.cast(pa.schema(fields))

# 전체 파일의 스키마를 먼저 확인
schemas = []

for path in paths:
    schema = pq.read_schema(path)

    schemas.append(pa.schema([
        pa.field(
            field.name,
            (
                field.type.value_type
                if pa.types.is_dictionary(field.type)
                else field.type
            ),
            nullable=True,
        )
        for field in schema
    ]))

columns = schemas[0].names

for path, schema in zip(paths, schemas):
    if set(schema.names) != set(columns):
        raise ValueError(
            f"열 구성이 다릅니다: {path.name}\n"
            "공간 변수 추가가 모든 월에 완료됐는지 확인하세요."
        )

# null형 등 월별 자료형 차이를 통합
schema = pa.unify_schemas(
    schemas,
    promote_options="permissive",
)

total_rows = 0

with pq.ParquetWriter(
    temp_path,
    schema,
    compression="snappy",
) as writer:

    for i, path in enumerate(paths, start=1):
        table = normalize_table(path)
        table = table.select(schema.names).cast(schema)

        writer.write_table(table)
        total_rows += table.num_rows

        print(
            f"[{i:02d}/40] {path.name} | "
            f"누적 {total_rows:,}행",
            flush=True,
        )

        del table

# 저장된 행 수 검증 후 최종 파일명으로 변경
saved_rows = pq.ParquetFile(temp_path).metadata.num_rows
assert saved_rows == total_rows

temp_path.replace(output_path)

print("\n저장 완료")
print("파일:", output_path)
print(f"전체 행 수: {saved_rows:,}")
print(f"파일 크기: {output_path.stat().st_size / 1024**2:,.1f} MB")